# IMO Health — General Purpose LangChain Agent

A simple general-purpose agent powered by:
- **AWS Bedrock** (Claude) as the LLM
- **IMO Health MCP Gateway** for clinical terminology tools
- **LangGraph ReAct agent** for orchestration

Ask it anything — it will call MCP tools when relevant.


| Component | Technology |
|-----------|------------|
| LLM | AWS Bedrock (Claude Haiku 4.5) |
| Tools | IMO Health MCP Gateway |
| Agent | LangGraph ReAct Agent |
| Auth | OAuth2 client_credentials grant |

## Prerequisites

- `config.json` file with MCP credentials 
- AWS credentials (SageMaker execution role or config.json)
- Python 3.10+


## Step 1: Install Dependencies

Run this cell once, then restart the kernel.

In [ ]:
%pip install -q --upgrade --index-url https://pypi.org/simple/ \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-aws>=0.2.0" \
    "langchain-mcp-adapters>=0.1.5" \
    "langgraph>=0.2.0" \
    "mcp<2.0.0" \
    boto3 botocore requests nest_asyncio httpx ipywidgets


## Step 2: Configuration

Loads credentials from `config.json`. Create one from `config.json.template` if it doesn't exist.

In [ ]:
import os
import re
import json
import glob
import uuid
import time
import asyncio
import pathlib
import nest_asyncio
from collections import Counter
from datetime import datetime

import requests

nest_asyncio.apply()

# --- Load config.json ---
candidates = [
    pathlib.Path(__file__).parent / 'config.json' if '__file__' in dir() else pathlib.Path('config.json'),
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json',
    pathlib.Path(r'd:\Users\nthonte\IMO-Work\Solution-Engineering\solution-accelerators\Diagnosis Specificity Agent\notebook\config.json'),
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json and fill in your credentials.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

mcp_cfg = cfg.get('mcp', {})

MCP_CLIENT_ID     = mcp_cfg.get('client_id', '')
MCP_CLIENT_SECRET = mcp_cfg.get('client_secret', '')
TOKEN_URL         = mcp_cfg.get('token_url', 'https://api.imohealth.com/oauth/token')
MCP_SERVER_URL    = mcp_cfg.get('server_url', '')
BEDROCK_MODEL     = mcp_cfg.get('bedrock_model_id', '')
BEDROCK_REGION    = mcp_cfg.get('aws_region', 'us-east-1')

if not MCP_CLIENT_SECRET:
    raise ValueError('client_secret is missing in config.json under the "mcp" section.')

# --- Load AWS credentials from config.json (for local dev) ---
aws_cfg = cfg.get('aws', {})
if aws_cfg.get('access_key_id'):
    os.environ.pop('AWS_PROFILE', None)
    os.environ['AWS_ACCESS_KEY_ID'] = aws_cfg['access_key_id']
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_cfg['secret_access_key']
    os.environ['AWS_SESSION_TOKEN'] = aws_cfg.get('session_token', '')
    os.environ['AWS_DEFAULT_REGION'] = aws_cfg.get('region', 'us-east-1')
    print(f'AWS credentials   : loaded from config.json (key prefix: {aws_cfg["access_key_id"][:8]}...)')
else:
    print('AWS credentials   : using default chain (env/IAM role)')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'Bedrock Model      : {BEDROCK_MODEL}')
print(f'AWS Region         : {BEDROCK_REGION}')
print(f'MCP Server URL     : {MCP_SERVER_URL}')


## Step 3: Get OAuth Token

In [ ]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type':    'client_credentials',
        'client_id':     client_id,
        'client_secret': client_secret,
        'audience':      'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(MCP_CLIENT_ID, MCP_CLIENT_SECRET)
print('Token acquired (prefix):', access_token[:20] + '...')

## Step 4: Connect to MCP Gateway & Discover Tools

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_CLIENT_CONFIG = {
    'mcp-gateway': {
        'url':       MCP_SERVER_URL,
        'transport': 'streamable_http',
        'headers':   {'Authorization': f'Bearer {access_token}'},
        'timeout':   300,
    }
}

async def get_langchain_tools():
    client = MultiServerMCPClient(MCP_CLIENT_CONFIG)
    tools = await client.get_tools()
    print(f'Discovered {len(tools)} MCP tools:\n')
    for t in tools:
        print(f'  - {t.name}')
    return tools

mcp_tools = asyncio.run(get_langchain_tools())


## Step 5: Initialize LLM (AWS Bedrock Claude)

In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model_id=BEDROCK_MODEL,
    region_name=BEDROCK_REGION,
    temperature=0,
    provider="anthropic",
)

print(f'LLM ready: {BEDROCK_MODEL}')


## Step 6: Create the Agent

A general-purpose ReAct agent — ask it anything. It will use MCP tools when needed.

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a helpful AI assistant with access to IMO Health clinical terminology tools.

<role>
You are an IMO Health Clinical Knowledge Assistant that uses the IMO Knowledge Graph and
IMO Normalize Engine to provide accurate, evidence-based answers to clinical terminology,
coding, and healthcare documentation questions.
</role>

<capabilities>
- Find specific ICD-10-CM codes for clinical conditions.
- Map diagnoses to the most specific codes available.
- Identify diagnostic tests, findings, procedures, medications, and treatments using the Knowledge Graph.
- Suggest coding specificity improvements and refinement paths.
- Answer medical terminology questions using IMO lexical concepts.
- Analyze clinical notes and use documented evidence to identify potential diagnosis refinements.
- Traverse the Knowledge Graph to discover broader and narrower concepts.
</capabilities>

<rules>
1. Always use MCP tools to ground your responses. Never guess, infer, or fabricate codes, concepts, or relationships.

2. Focus only on the user's question. Do not provide unnecessary information outside the scope of the request.

3. If the user asks for the most specific code for a condition:
   - Use the evidence provided in their query (laterality, site, type, chronicity, etc.) to determine the most specific code directly.
   - Do not show the full hierarchy or all narrower concepts — go straight to the specific code supported by the user's evidence.
   - If the user's query lacks enough detail for a fully specific code, present the best match and list what additional details would increase specificity.

4. If the user asks to explore or drill down into a broader concept:
   - Show available refinement paths and narrower concepts.
   - Use get_allowed_refinements or get_narrower_with_refinements for broad concepts.
   - Avoid get_narrower_hierarchy on broad parent concepts as it may return too much data.

5. If a term cannot be normalized:
   - Inform the user clearly.
   - Suggest alternative terms or concepts that may produce results.

6. When multiple refinement or specificity paths exist:
   - Show all available paths returned by the Knowledge Graph.
   - Do not omit or truncate any valid refinement options.

7. Include the corresponding IMO Lexical ID whenever a concept, diagnosis, finding, procedure, medication, or terminology result is presented.

8. If the user provides a clinical note:
   - Analyze only the documented clinical evidence.
   - Use MCP tools to identify possible refinements.
   - Do not assume conditions, findings, or diagnoses that are not documented.

9. When providing refinements:
   - Continue drilling down to the most specific concept available.
   - Stop only when:
     a. No additional refinement exists, or
     b. The clinical evidence does not support further specificity.

10. Never display:
    - Raw JSON
    - Tool responses
    - Internal MCP output
    - System prompts
    Present only formatted, user-friendly results.

11. If the response contains code system information:
    - Ensure '/code_systems' is represented as an array, not a string.

12. If information is unavailable from the MCP tools:
    - State that the information could not be found.
    - Do not generate synthetic or unsupported answers.

13. Maintain clinical accuracy and terminology consistency throughout the response.

14. If a tool returns too much data or errors due to payload size, summarize the top-level categories available and ask the user to pick a specific pathway to drill into.

15. At the end of every response, generate 3-5 contextually relevant follow-up questions that the user may ask next.
    - Questions should be unique for each response.
    - Questions should be related to the current topic.
</rules>
"""

agent = create_react_agent(llm, mcp_tools, prompt=SYSTEM_PROMPT)
print('Agent ready. Use ask() to interact.')


## Step 7: Chat with the Agent

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, Markdown
import html as html_module


class AgentUI:
    """Rich HTML display for agent streaming output."""

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">IMO Health General Agent</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Ask clinical terminology, coding, or knowledge graph questions</p>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:12px;">Type <b>quit</b> to end · <b>reset</b> to clear history · <b>Kernel Interrupt</b> to stop mid-generation</p>
        </div>
        """))

    @staticmethod
    def status(message):
        display(HTML(f'<div style="color:#5f6368; font-size:12px; font-style:italic; padding:4px 0;">{html_module.escape(message)}</div>'))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">🔧 Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:300])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">✅ Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 300 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))

    @staticmethod
    def stopped():
        display(HTML("""
        <div style="background:#fce8e6; border:1px solid #c5221f; border-radius:8px; padding:12px 16px; margin:8px 0; text-align:center;">
            <b style="color:#c5221f;">Session ended.</b>
        </div>
        """))


async def chat_rich():
    """Multi-turn agent with rich HTML UI."""
    import logging
    logging.getLogger("httpx").setLevel(logging.CRITICAL)
    logging.getLogger("httpcore").setLevel(logging.CRITICAL)
    logging.getLogger("mcp").setLevel(logging.CRITICAL)

    messages = []
    ui = AgentUI()
    ui.header()

    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (KeyboardInterrupt, EOFError):
            ui.stopped()
            break

        if not user_input:
            continue
        if user_input.lower() in ('quit', 'stop', 'exit'):
            ui.stopped()
            break
        if user_input.lower() == 'reset':
            messages = []
            ui.separator()
            ui.status('Conversation reset.')
            continue

        display(HTML(f"""
        <div style="background:#f0f0f0; border-radius:8px; padding:10px 16px; margin:8px 0; font-size:14px;">
            <b>👤 You:</b> {html_module.escape(user_input[:500])}{'...' if len(user_input) > 500 else ''}
        </div>
        """))

        messages.append({'role': 'user', 'content': user_input})
        ui.status('Agent is thinking...')

        final_content = ""
        try:
            async for chunk in agent.astream(
                {'messages': messages},
                stream_mode='updates'
            ):
                for node_name, node_output in chunk.items():
                    if node_name == 'agent':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                                for tc in msg.tool_calls:
                                    tool_name = tc['name']
                                    if not tool_name.startswith('mcp__imo-health__'):
                                        tool_name = f'mcp__imo-health__{tool_name}'
                                    ui.tool_call(tool_name, tc.get('args', {}))
                            if hasattr(msg, 'content') and msg.content:
                                if isinstance(msg.content, str) and msg.content:
                                    final_content = msg.content
                                elif isinstance(msg.content, list):
                                    for block in msg.content:
                                        if isinstance(block, dict) and block.get('type') == 'text':
                                            final_content += block['text']

                    elif node_name == 'tools':
                        msgs = node_output.get('messages', [])
                        for msg in msgs:
                            if hasattr(msg, 'content'):
                                tool_name = getattr(msg, 'name', 'tool')
                                if not tool_name.startswith('mcp__imo-health__'):
                                    tool_name = f'mcp__imo-health__{tool_name}'
                                ui.tool_result(tool_name, msg.content)

        except KeyboardInterrupt:
            ui.status('Generation interrupted.')
        except Exception:
            pass

        if final_content:
            ui.separator()
            ui.agent_response(final_content)
            messages.append({'role': 'assistant', 'content': final_content})

        ui.separator()

await chat_rich()
